# Analisis Downstream XLM-RoBERTa - NusaX-Senti

Paralel dengan `finetune_nusabert.ipynb`. Menganalisis hasil `finetune_xlmr.py`:
- **A.** Delta-F1 augmented vs baseline (7 bahasa target)
- **B.** Sanity-check baseline vs NusaX Tabel 2 (reproduksi)
- **C.** Training curves (loss + val F1) per bahasa
- **D.** Confusion matrix + classification report per bahasa
- **E.** Delta F1 per kelas (efek augmentasi per kelas sentimen)

Ganti `SIZE` ("large"/"base") untuk model lain. Cell D & E butuh GPU.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path("y:/Michh/Python/Projects/MAGenerator")
SIZE = "large"   # "large" atau "base"
MODEL_NAME = f"xlm-roberta-{SIZE}"
BASE_DIR = ROOT / f"outputs/xlmr-sentiment-{SIZE}"       # baseline (12 bahasa)
AUG_DIR  = ROOT / f"outputs/xlmr-sentiment-{SIZE}-syn"   # augmented (7 bahasa)
DATA_DIR = ROOT / "data/nusax_senti"
TARGET_LANGS = ["ace", "ban", "bjn", "jav", "mad", "min", "sun"]
CLASSES = ["negative", "neutral", "positive"]

NUSAX_T2 = {
    "large": {"ace":75.9,"ban":77.1,"bbc":65.5,"bjn":86.3,"bug":70.0,"eng":92.6,"ind":91.6,"jav":84.2,"mad":74.9,"min":83.1,"nij":73.3,"sun":86.0},
    "base":  {"ace":73.9,"ban":72.8,"bbc":62.3,"bjn":76.6,"bug":66.6,"eng":90.8,"ind":88.4,"jav":78.9,"mad":69.7,"min":79.1,"nij":75.0,"sun":80.1},
}[SIZE]

def load_summary(d):
    p = d / "results_summary.json"
    return json.load(open(p))["results"] if p.exists() else None

base_res = load_summary(BASE_DIR)
aug_res  = load_summary(AUG_DIR)
print("Model :", MODEL_NAME)
print("Baseline :", "FOUND" if base_res else "MISSING", "->", BASE_DIR)
print("Augmented:", "FOUND" if aug_res else "MISSING", "->", AUG_DIR)

## A. Delta-F1 Augmented vs Baseline (7 bahasa target)

In [ ]:
if base_res and aug_res:
    rows = []
    for lang in TARGET_LANGS:
        b = base_res.get(lang, {}).get("test_f1")
        a = aug_res.get(lang, {}).get("test_f1")
        if b is None or a is None:
            continue
        rows.append({"lang": lang, "baseline": round(b*100, 2), "augmented": round(a*100, 2),
                     "delta": round((a-b)*100, 2)})
    df = pd.DataFrame(rows)
    if len(df):
        mean_row = {"lang": "MEAN", "baseline": round(df["baseline"].mean(), 2),
                    "augmented": round(df["augmented"].mean(), 2), "delta": round(df["delta"].mean(), 2)}
        df = pd.concat([df, pd.DataFrame([mean_row])], ignore_index=True)
        display(df)
        d = df[df["lang"] != "MEAN"]["delta"]
        print(f"Winners (d>0): {(d>0).sum()}/{len(d)}  |  Losers (d<0): {(d<0).sum()}/{len(d)}")
else:
    print("Run baseline + augmented dulu (finetune_xlmr.py).")

## B. Sanity-Check Baseline vs NusaX Tabel 2

Baseline harus dalam +/-2pt dari NusaX (toleransi non-determinisme). Banyak CHECK -> recipe ditinjau.

In [ ]:
if base_res:
    rows = []
    for lang in sorted(base_res.keys()):
        b = base_res[lang]["test_f1"] * 100
        ref = NUSAX_T2.get(lang)
        dev = (b - ref) if ref is not None else None
        rows.append({"lang": lang, "our_baseline": round(b, 2), "nusax_T2": ref,
                     "deviation": round(dev, 2) if dev is not None else None,
                     "flag": ("OK" if abs(dev) <= 2 else "CHECK") if dev is not None else ""})
    df = pd.DataFrame(rows)
    display(df)
    valid = df[df["flag"].isin(["OK", "CHECK"])]
    print(f"Reproduksi: {(valid['flag']=='OK').sum()}/{len(valid)} dalam +/-2pt | "
          f"mean abs(deviation) = {valid['deviation'].abs().mean():.2f} pt")
    print("Deviasi wajar krn non-determinisme (seed/GPU/versi) + NusaX kemungkinan rata-rata banyak seed.")
else:
    print("Run baseline dulu.")

## C. Training Curves per Bahasa (loss + val F1)

Ganti `SRC_DIR` ke BASE_DIR atau AUG_DIR.

In [ ]:
SRC_DIR = AUG_DIR   # AUG_DIR (augmented) atau BASE_DIR (baseline)
avail = [l for l in TARGET_LANGS if (SRC_DIR / f"{MODEL_NAME}-{l}" / "train_history.json").exists()]
if avail:
    n = len(avail)
    fig, axes = plt.subplots(n, 2, figsize=(12, 4*n))
    if n == 1:
        axes = axes.reshape(1, 2)
    for r, lang in enumerate(avail):
        h = json.load(open(SRC_DIR / f"{MODEL_NAME}-{lang}" / "train_history.json"))
        tr = [(e["epoch"], e["loss"]) for e in h if "loss" in e and "eval_loss" not in e]
        ev = {e["epoch"]: e for e in h if "eval_loss" in e}
        ev = sorted(ev.values(), key=lambda x: x["epoch"])
        ax1 = axes[r, 0]
        ax1.plot([e for e, _ in tr], [l for _, l in tr], ".-", label="train loss")
        ax1.plot([e["epoch"] for e in ev], [e["eval_loss"] for e in ev], ".-", label="val loss")
        ax1.set_title(f"{lang.upper()} - loss"); ax1.set_xlabel("epoch"); ax1.legend(); ax1.grid(alpha=.2)
        ax2 = axes[r, 1]
        vf = [e["eval_f1"]*100 for e in ev]
        ax2.plot([e["epoch"] for e in ev], vf, ".-", color="tab:blue", label="val F1")
        bi = int(np.argmax(vf))
        ax2.plot(ev[bi]["epoch"], vf[bi], "o", color="green", markersize=7, label=f"best {vf[bi]:.1f}% (ep {ev[bi]['epoch']:.0f})")
        ax2.set_title(f"{lang.upper()} - val F1"); ax2.set_xlabel("epoch"); ax2.legend(); ax2.grid(alpha=.2)
    plt.suptitle(f"{MODEL_NAME} - {SRC_DIR.name}", fontsize=13, fontweight="bold")
    plt.tight_layout(); plt.show()
else:
    print(f"Belum ada train_history di {SRC_DIR}. Run dulu.")

## D. Confusion Matrix + Classification Report per Bahasa

Butuh GPU. Ganti `ARM_DIR` ke BASE_DIR/AUG_DIR.

In [ ]:
import torch
from transformers import pipeline
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

ARM_DIR = AUG_DIR   # AUG_DIR atau BASE_DIR
device = 0 if torch.cuda.is_available() else -1
avail = [l for l in TARGET_LANGS if (ARM_DIR / f"{MODEL_NAME}-{l}" / "best").exists()]
preds = {}
for lang in avail:
    mp = ARM_DIR / f"{MODEL_NAME}-{lang}" / "best"
    clf = pipeline("text-classification", model=str(mp), tokenizer=str(mp), device=device, top_k=None)
    test = pd.read_csv(DATA_DIR / lang / "test.csv")
    raw = clf(test["text"].tolist(), batch_size=64)
    pl = [max(p, key=lambda x: x["score"])["label"] for p in raw]
    preds[lang] = {"true": test["label"].tolist(), "pred": pl}
    print(f"\n=== {lang.upper()} ({ARM_DIR.name}) ===")
    print(classification_report(preds[lang]["true"], pl, labels=CLASSES, digits=3))
    del clf
    torch.cuda.empty_cache()

if preds:
    n = len(preds); cols = 4; rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 3.5*rows))
    axes = np.array(axes).flatten()
    for i, (lang, d) in enumerate(preds.items()):
        cm = confusion_matrix(d["true"], d["pred"], labels=CLASSES)
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                    xticklabels=[c[:3] for c in CLASSES], yticklabels=[c[:3] for c in CLASSES],
                    ax=axes[i], cbar=False)
        axes[i].set_title(lang.upper()); axes[i].set_xlabel("pred"); axes[i].set_ylabel("true")
    for j in range(len(preds), len(axes)):
        axes[j].axis("off")
    plt.suptitle(f"{MODEL_NAME} confusion - {ARM_DIR.name}", fontsize=13, fontweight="bold")
    plt.tight_layout(); plt.show()
else:
    print(f"Belum ada model di {ARM_DIR}. Run dulu.")

## E. Delta F1 per Kelas (augmented - baseline)

Butuh KEDUA arm selesai. Tahu kelas mana yang dibantu/dirugikan augmentasi (mis. neutral).

In [ ]:
import torch
from transformers import pipeline
from sklearn.metrics import f1_score
device = 0 if torch.cuda.is_available() else -1

def predict_arm(arm_dir, lang):
    mp = arm_dir / f"{MODEL_NAME}-{lang}" / "best"
    if not mp.exists():
        return None
    clf = pipeline("text-classification", model=str(mp), tokenizer=str(mp), device=device, top_k=None)
    test = pd.read_csv(DATA_DIR / lang / "test.csv")
    raw = clf(test["text"].tolist(), batch_size=64)
    pl = [max(p, key=lambda x: x["score"])["label"] for p in raw]
    del clf
    torch.cuda.empty_cache()
    return test["label"].tolist(), pl

rows = []
for lang in TARGET_LANGS:
    rb = predict_arm(BASE_DIR, lang)
    ra = predict_arm(AUG_DIR, lang)
    if rb is None or ra is None:
        continue
    for c in CLASSES:
        fb = f1_score(rb[0], rb[1], labels=[c], average="macro") * 100
        fa = f1_score(ra[0], ra[1], labels=[c], average="macro") * 100
        rows.append({"lang": lang, "class": c, "base_F1": round(fb, 1), "aug_F1": round(fa, 1), "delta": round(fa-fb, 1)})

if rows:
    df = pd.DataFrame(rows)
    print("Delta F1 per kelas (augmented - baseline). Positif = augmentasi membantu kelas itu.")
    display(df.pivot(index="lang", columns="class", values="delta"))
    print("\nMean Delta per kelas:")
    display(df.groupby("class")["delta"].mean().round(2))
else:
    print("Butuh KEDUA arm selesai untuk semua bahasa.")